# v14 · ValaQuenta `hyperwebster`: Horner index + Fano path + `SemanticWord`
`ValaQuenta/modules/hyperwebster/maths.py` (git 2026-06-28). The same construction as v00 (`a–z A–Z 0–9…` order, base-97 mixed radix, `char_index % 7` Fano path), packaged as a module.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
vq = load("ValaQuenta/modules/hyperwebster/maths.py", "vq")
w = vq.SemanticWord("hello")
print(w.horner_idx, w.fano_idx, w.length, w.verify())
assert vq.horner_decode(vq.horner_encode("Hello, World!"), 13) == "Hello, World!"

623460239 1596 5 True


## Not a bijection over strings: the length is part of the address
Symbol 0 is `a`; `a`, `aa`, `aaa` all have index 0.

In [3]:
print({k: vq.horner_encode("a"*k) for k in (1, 2, 3)})
assert vq.horner_encode("a") == vq.horner_encode("aaa") == 0

{1: 0, 2: 0, 3: 0}


## An unknown character silently becomes symbol 0

In [4]:
print("é ->", vq.char_to_idx("é"), "| horner_encode('é') == horner_encode('a'):", vq.horner_encode("é") == vq.horner_encode("a"))
assert vq.horner_encode("é") == vq.horner_encode("a")
# v02 (bijective, shipped) rejects it instead
hw = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw").HyperWebster()
try: hw.point_to_text("é")
except ValueError as e: print("v02:", e)

é -> 0 | horner_encode('é') == horner_encode('a'): True
v02: Character 'é' is not in the character set.


## The Fano index: seven classes

In [5]:
import re
words = sorted(set(re.findall(r"[a-z]+", CORPUS.lower())))
fano = Counter(vq.fano_encode(w) for w in words)
print(len(words), "distinct words ->", len(fano), "distinct Fano indices | largest class:", max(fano.values()))
sw = [vq.SemanticWord("listen"), vq.SemanticWord("silent")]
print("anagrams: horner differ:", sw[0].horner_idx != sw[1].horner_idx, "| fano differ:", sw[0].fano_idx != sw[1].fano_idx)

206 distinct words -> 202 distinct Fano indices | largest class: 2
anagrams: horner differ: True | fano differ: True


The Fano index keeps the sequence of `index mod 7` values, so it distinguishes anagrams (order is kept) but merges symbols that are congruent mod 7. The Horner index is exact once the length is stored and the text stays inside the 97 symbols.